# loudkit quickstart

Text to speech that runs on your own machine, in ten languages, with voice
cloning from ten seconds of audio.

This notebook downloads the model, speaks a sentence, and clones a voice.
Use Runtime → Run all, or run the cells one by one. A GPU runtime is faster,
and the CPU also works. The audio can differ slightly between the two.

[Repository](https://github.com/loudreader/loudkit) ·
[Voices](https://github.com/loudreader/loudkit/blob/main/VOICES.md) ·
[Model card](https://github.com/loudreader/loudkit/blob/main/docs/MODEL_CARD.md)

In [ ]:
# The `hub` extra lets `load()` take a model name instead of a path.
%pip install -q "loudkit[torch,audio,hub]==0.1.1"

## Speak

The first call downloads about 750 MB and caches it. Later sessions on the
same machine skip the download, but still load the model. Load the engine
once and keep it. Cloning later adds the separate 523 MB enrollment checkpoint.

All 28 voices are included in both model downloads. Choose Henry, Oliver, Sophie or Oscar by name, for example `engine.voice("henry")`.


In [ ]:
from IPython.display import Audio

import loudkit as lk

MODEL = "loudreader/loudr-1"

engine = lk.load(MODEL)
narrator = engine.voice("joe")

result = engine.synthesize("Hello from a machine you control.", narrator, seed=7)
print(result)
Audio(result.audio, rate=result.sample_rate)

## The same seed gives the same audio

On one engine, the same text, voice and seed give identical samples. The
engine refuses to start if two of its components disagree about what to
compute. The five implementations (Python, Swift, Go, Rust, TypeScript) are
tested against one shared conformance fixture.

A different seed gives a different reading of the same text.

In [ ]:
import numpy as np

again = engine.synthesize("Hello from a machine you control.", narrator, seed=7)
print("bit-identical:", np.array_equal(result.audio, again.audio))

different = engine.synthesize("Hello from a machine you control.", narrator, seed=8)
print(
    "a different seed is a different reading:",
    not np.array_equal(result.audio, different.audio),
)
Audio(different.audio, rate=different.sample_rate)

## A whole passage

One window holds about ten seconds of speech. `synthesize` splits longer text
at sentence boundaries and carries a short token prefix across each join, so
each chunk continues from the one before. `stream` is the same synthesis
delivered chunk by chunk, so you can start playing before it finishes.

In [ ]:
passage = (
    "The first sentence sets the scene and runs on for a while. "
    "The second follows it and is no shorter than the first one was. "
    "The third exists so that the splitter has somewhere to breathe."
)
long = engine.synthesize(passage, narrator, seed=7)
print(f"{long.duration:.1f}s from {len(long.tokens)} tokens")
Audio(long.audio, rate=long.sample_rate)

## The faster model

`loudreader/loudr-1-turbo` decodes two tokens a step instead of one and
renders in a single flow step. It is faster end to end; how much depends on
the device and the passage (see the
[benchmarks](https://github.com/loudreader/loudkit/blob/main/docs/benchmarks.md)).
It carries the same 28 voices, and a voice profile made with one model works
with the other without conversion.

The next cell reads the same text in the same voice with the same seed on
both models. Listen to both and compare.


In [ ]:
import time

turbo = lk.load("loudreader/loudr-1-turbo")

for name, this in (("loudr-1", engine), ("loudr-1-turbo", turbo)):
    started = time.perf_counter()
    spoken = this.synthesize(passage, narrator, seed=7)
    elapsed = time.perf_counter() - started
    print(
        f"{name:14} {spoken.duration:5.2f}s of audio in {elapsed:5.2f}s"
        f"  ({spoken.duration / elapsed:.2f}x real time)"
    )
    display(Audio(spoken.audio, rate=spoken.sample_rate))

## Other languages

Voices ship for ten languages. **Only English quality has been evaluated.**
Listen to the other languages before you use them.

In [ ]:
gosia = lk.voice("gosia", repo=MODEL)
polish = engine.synthesize(
    "Pobierz aplikację i posłuchaj, jak brzmi ten głos po polsku.",
    gosia,
    seed=7,
    language="pl",
)
Audio(polish.audio, rate=polish.sample_rate)

## Clone a voice

Enroll a voice from about ten seconds of clean audio. The result is a voice
profile of about 150 KB. Enrollment does not train a new model.

**Get the speaker's consent before you clone their voice.** See
[RESPONSIBLE_USE.md](https://github.com/loudreader/loudkit/blob/main/RESPONSIBLE_USE.md).

In [ ]:
%pip install -q "loudkit[enroll]==0.1.1"

# Upload a recording with the Files pane on the left and point `SOURCE` at it.
# Use about 10 s of clean speech from one speaker.
SOURCE = None  # e.g. "/content/my-recording.wav"

if SOURCE:
    mine = lk.enroll(SOURCE, MODEL, name="my-voice")
    mine.save("my-voice.safetensors")

    spoken = engine.synthesize("This is my own voice, running locally.", mine, seed=7)
    display(Audio(spoken.audio, rate=spoken.sample_rate))
else:
    print("Set SOURCE to a recording to try cloning.")

## Where to go next

- Other programming languages: the Swift, Go, Rust and TypeScript ports are
  independent implementations, tested against the same conformance fixture. See
  [`docs/guides/`](https://github.com/loudreader/loudkit/tree/main/docs/guides).
- A local server: `loudkit serve` keeps the model loaded and answers HTTP;
  `loudkit serve --mcp` does the same for agents.
- How the implementations compare: [the measured parity
  table](https://github.com/loudreader/loudkit/blob/main/docs/parity-measured.md).